# <font color='#E7485C'> **Descripción del problema** </font>

El departamento de ventas de la empresa Holafly, cuenta con 60 colaboradores distribuidos entre tres mercados (América, Europa y Asia). Cada día de una semana de planificación, los colaboradores pueden ser asignados a uno de tres turnos de trabajo (mañana, tarde o noche), cada uno con una duración de ocho horas. El objetivo es diseñar un modelo de optimización que determine la asignación diaria de turnos para cada colaborador, de manera que la disponibilidad de personal se ajuste lo mejor posible al personal requerido en cada mercado y turno, respetando las restricciones laborales y de disponibilidad de los colaboradores.

A continuación, se definen explícitamente los elementos que deben considerarse en la modelación:


### <font color='#cc7285'> **Conjuntos** </font>
- Colaboradores: conjunto de empleados que deben asignarse
- Mercados: América, Europa, Asia
- Turnos de trabajo: Mañana (6:00-14:00), tarde (14:00-22:00), noche (22:00-6:00)
- Días de la semana de planificación (lunes, martes, miércoles, jueves, viernes, sábado, domingo)

### <font color='#cc7285'> **Parámetros** </font>
- Personal requerido por día, turno y mercado.
- Disponibilidad de cada colaborador para trabajar en un día y turno determinados.

### <font color='#cc7285'> **Decisiones** </font>
- Turno y mercado asignado a cada colaborador cada día

### <font color='#cc7285'> **Objetivo** </font>
- Minimizar la desviación absoluta entre el personal asignado y el personal requerido.

### <font color='#cc7285'> **Restricciones** </font>
- Cada colaborador puede ser asignado, como máximo, a un turno por día.
- Las asignaciones deben respetar la disponibilidad de cada colaborador.
- Un colaborador que trabaje en el turno de noche no puede ser asignado al turno de mañana del día siguiente.
- Cada colaborador debe tener al menos un día completo de descanso durante la semana.

### <font color='#cc7285'> **Supuestos** </font>
- Cada colaborador puede atender cualquiera de los tres mercados

# <font color='#E7485C'> **Modelación** </font>

### <font color='#cc7285'> **Conjuntos** </font>
- $C:$ Colaboradores ($i$)
- $M:$ Mercados ($m$)
- $T:$ Turnos ($t$)
- $D:$ Días de la semana ($d$)

### <font color='#cc7285'> **Parámetros** </font>
- $R_{dtm}:$ Personal requerido el día $d$, turno $t$, mercado $m$
- $A_{idt}:$ 1 si el colaborador $i$ está disponible el día $i$ en el turno $t$


### <font color='#cc7285'> **Decisiones** </font>
- $x_{idtm}:$ 1 si el colaborador $i$ trabaja el día $d$, turno $t$, mercado $m$, 0 en otro caso

Variables auxiliares:
- $e_{dtm} \geq 0$: exceso de personal
- $f_{dtm} \geq 0$: déficit de personal

### <font color='#cc7285'> **Función objetivo** </font>
Minimizar la desviación absoluta entre el personal asignado y el requerido.

min $ \sum_{d \in D}\sum_{t \in T}\sum_{m \in M} (e_{dtm} + f_{dtm})$


### <font color='#cc7285'> **Restricciones** </font>
1. Un turno por colaborador al día: Cada colaborador puede trabajar como máximo un turno diario.

$$ \sum_{t \in T}\sum_{m \in M} x_{idtm}\leq 1 \forall i \in C, d \in D $$

2. Respetar disponibilidad: Un colaborador solo puede ser asignado si está disponible.
$$ \sum_{m \in M} x_{idtm}\leq A_{ids} \forall i \in C, d \in D, t\in T$$

3. Balance entre asignación y requerimiento: La diferencia entre personal asignado y requerido se representa mediante exceso y déficit.
$$ \sum_{i \in C} x_{idtm} - R_{dtm} = e_{dtm} - f_{dtm} \forall d \in D, t \in T, m\in M$$

4. Descanso después de turno de noche
$$ \sum_{m \in M} x_{id,t_N,m} + \sum_{m \in M} x_{i,d+1,t_M,m} \leq 1 \forall i, d = 1, ..., 7 $$

5. Mínimo de descanso semanal: Cada colaborador debe tener al menos un día sin asignación.
$$ \sum_{d \in D}\sum_{t \in T}\sum_{m \in M} x_{idtm}\leq 6 \forall i \in C $$

6. Tipo de variables
$$x_{idtm} \in {0,1} $$
$$e_{dtm} \geq 0$$
$$f_{dtm} \geq 0$$




# <font color='#E7485C'> **Implementación** </font>

In [2]:
pip install pyomo

Note: you may need to restart the kernel to use updated packages.


In [31]:
#Librerías
from pyomo.environ import *
import random
import pandas as pd
from pyomo.environ import value

In [16]:
colaboradores = [f"C{i}" for i in range(1, 61)]
dias = [
    "Lunes",
    "Martes",
    "Miércoles",
    "Jueves",
    "Viernes",
    "Sábado",
    "Domingo"
]
turnos = ["Mañana", "Tarde", "Noche"]
mercados = ["America", "Europa", "Asia"]

In [25]:
# Disponibilidad trabajadores
# Supondremos que cada colaborador tiene un 90% de probabilidad de estar disponible en un turno.

random.seed(10)
A = {}

for i in colaboradores:
    for d in dias:
        for t in turnos:
            A[(i,d,t)] = 1 if random.random() < 0.9 else 0

print(A)

{('C1', 'Lunes', 'Mañana'): 1, ('C1', 'Lunes', 'Tarde'): 1, ('C1', 'Lunes', 'Noche'): 1, ('C1', 'Martes', 'Mañana'): 1, ('C1', 'Martes', 'Tarde'): 1, ('C1', 'Martes', 'Noche'): 1, ('C1', 'Miércoles', 'Mañana'): 1, ('C1', 'Miércoles', 'Tarde'): 1, ('C1', 'Miércoles', 'Noche'): 1, ('C1', 'Jueves', 'Mañana'): 1, ('C1', 'Jueves', 'Tarde'): 1, ('C1', 'Jueves', 'Noche'): 0, ('C1', 'Viernes', 'Mañana'): 0, ('C1', 'Viernes', 'Tarde'): 1, ('C1', 'Viernes', 'Noche'): 1, ('C1', 'Sábado', 'Mañana'): 1, ('C1', 'Sábado', 'Tarde'): 1, ('C1', 'Sábado', 'Noche'): 1, ('C1', 'Domingo', 'Mañana'): 1, ('C1', 'Domingo', 'Tarde'): 1, ('C1', 'Domingo', 'Noche'): 1, ('C2', 'Lunes', 'Mañana'): 1, ('C2', 'Lunes', 'Tarde'): 1, ('C2', 'Lunes', 'Noche'): 1, ('C2', 'Martes', 'Mañana'): 0, ('C2', 'Martes', 'Tarde'): 0, ('C2', 'Martes', 'Noche'): 1, ('C2', 'Miércoles', 'Mañana'): 1, ('C2', 'Miércoles', 'Tarde'): 1, ('C2', 'Miércoles', 'Noche'): 1, ('C2', 'Jueves', 'Mañana'): 0, ('C2', 'Jueves', 'Tarde'): 1, ('C2', 'Ju

In [26]:
# Personal requerido
R = {}

for d in dias:
    for t in turnos:
        for m in mercados:
            R[(d,t,m)] = random.randint(4,10)

print(R)

{('Lunes', 'Mañana', 'America'): 10, ('Lunes', 'Mañana', 'Europa'): 5, ('Lunes', 'Mañana', 'Asia'): 7, ('Lunes', 'Tarde', 'America'): 10, ('Lunes', 'Tarde', 'Europa'): 4, ('Lunes', 'Tarde', 'Asia'): 8, ('Lunes', 'Noche', 'America'): 4, ('Lunes', 'Noche', 'Europa'): 4, ('Lunes', 'Noche', 'Asia'): 9, ('Martes', 'Mañana', 'America'): 5, ('Martes', 'Mañana', 'Europa'): 8, ('Martes', 'Mañana', 'Asia'): 6, ('Martes', 'Tarde', 'America'): 7, ('Martes', 'Tarde', 'Europa'): 7, ('Martes', 'Tarde', 'Asia'): 10, ('Martes', 'Noche', 'America'): 9, ('Martes', 'Noche', 'Europa'): 4, ('Martes', 'Noche', 'Asia'): 7, ('Miércoles', 'Mañana', 'America'): 7, ('Miércoles', 'Mañana', 'Europa'): 4, ('Miércoles', 'Mañana', 'Asia'): 4, ('Miércoles', 'Tarde', 'America'): 10, ('Miércoles', 'Tarde', 'Europa'): 6, ('Miércoles', 'Tarde', 'Asia'): 7, ('Miércoles', 'Noche', 'America'): 4, ('Miércoles', 'Noche', 'Europa'): 9, ('Miércoles', 'Noche', 'Asia'): 10, ('Jueves', 'Mañana', 'America'): 5, ('Jueves', 'Mañana', '

In [27]:
# Crear modelo
model = ConcreteModel()

# CONJUNTOS
model.C = Set(initialize=colaboradores)   # Colaboradores
model.D = Set(initialize=dias)   # Días
model.T = Set(initialize=turnos)   # Turnos
model.M = Set(initialize=mercados)   # Mercados


# PARÁMETROS
# Personal requerido
model.R = Param(model.D, model.T, model.M, initialize=R, within=NonNegativeIntegers)

# Disponibilidad
model.A = Param(model.C, model.D, model.T, initialize=A, within=Binary)


# VARIABLES DE DECISIÓN
# Asignación
model.x = Var(model.C, model.D, model.T, model.M, within=Binary)

# Exceso
model.e = Var(model.D, model.T, model.M, within=NonNegativeReals)

# Déficit
model.f = Var(model.D, model.T, model.M, within=NonNegativeReals)

# FUNCIÓN OBJETIVO
def objective_rule(model):
    return sum(
        model.e[d,t,m] + model.f[d,t,m]
        for d in model.D
        for t in model.T
        for m in model.M
    )
model.obj = Objective(rule=objective_rule,sense=minimize)


# RESTRICCIONES
# Un turno por colaborador por día
def one_shift_rule(model, i, d):
    return sum(
        model.x[i,d,t,m]
        for t in model.T
        for m in model.M
    ) <= 1
model.one_shift = Constraint(model.C,model.D,rule=one_shift_rule)


# Disponibilidad
def availability_rule(model, i, d, t):
    return sum(
        model.x[i,d,t,m]
        for m in model.M
    ) <= model.A[i,d,t]
model.availability = Constraint(model.C, model.D, model.T, rule=availability_rule)

# Balance entre asignados y requeridos
def balance_rule(model, d, t, m):
    return (
        sum(model.x[i,d,t,m] for i in model.C)
        - model.R[d,t,m]
        ==
        model.e[d,t,m] - model.f[d,t,m]
    )
model.balance = Constraint(model.D, model.T, model.M, rule=balance_rule)


# Descanso después de turno noche
dias = list(model.D)
def night_rest_rule(model, i, idx):

    if idx == len(dias)-1:
        return Constraint.Skip

    d = dias[idx]
    d_sig = dias[idx+1]

    return (sum(model.x[i,d,"Noche",m] for m in model.M) +
        sum(model.x[i,d_sig,"Mañana",m] for m in model.M) <= 1)
model.night_rest = Constraint(model.C, RangeSet(0, len(dias)-1), rule=night_rest_rule)


# Al menos un día libre
def weekly_rest_rule(model, i):
    return sum(
        model.x[i,d,t,m]
        for d in model.D
        for t in model.T
        for m in model.M
    ) <= 6
model.weekly_rest = Constraint(model.C, rule=weekly_rest_rule)

In [ ]:
pip install highspy

Note: you may need to restart the kernel to use updated packages.


In [28]:
solver = SolverFactory('appsi_highs')  
results = solver.solve(model)

In [29]:
print("Valor objetivo:", value(model.obj))

for i in model.C:
    for d in model.D:
        for t in model.T:
            for m in model.M:
                if value(model.x[i,d,t,m]) > 0.5:
                    print(i, d, t, m)

Valor objetivo: 87.0
C1 Lunes Mañana Asia
C1 Martes Tarde Asia
C1 Miércoles Mañana Asia
C1 Viernes Tarde Asia
C1 Sábado Mañana Europa
C1 Domingo Mañana Asia
C2 Lunes Mañana Europa
C2 Martes Noche Asia
C2 Jueves Noche Asia
C2 Viernes Noche Europa
C2 Sábado Noche Europa
C2 Domingo Tarde Asia
C3 Lunes Noche America
C3 Martes Noche Asia
C3 Miércoles Tarde Asia
C3 Jueves Mañana Asia
C3 Viernes Noche America
C3 Domingo Noche Europa
C4 Lunes Noche Europa
C4 Martes Tarde America
C4 Jueves Noche Europa
C4 Viernes Tarde Europa
C4 Sábado Mañana Asia
C4 Domingo Noche Europa
C5 Lunes Mañana Asia
C5 Martes Tarde Asia
C5 Miércoles Noche Asia
C5 Viernes Mañana Europa
C5 Sábado Noche Asia
C5 Domingo Tarde Asia
C6 Lunes Noche America
C6 Miércoles Mañana America
C6 Jueves Noche Europa
C6 Viernes Tarde Asia
C6 Sábado Tarde America
C6 Domingo Tarde America
C7 Lunes Tarde Asia
C7 Martes Tarde Asia
C7 Miércoles Noche Asia
C7 Jueves Tarde Asia
C7 Viernes Noche America
C7 Sábado Tarde Asia
C8 Lunes Tarde Ameri

In [34]:


# Lista para almacenar las asignaciones
asignaciones = []

for i in model.C:
    for d in model.D:
        for t in model.T:
            for m in model.M:
                if value(model.x[i, d, t, m]) > 0.5:
                    asignaciones.append({
                        "Colaborador": i,
                        "Día": d,
                        "Turno": t,
                        "Mercado": m
                    })

# Crear DataFrame
df_asignaciones = pd.DataFrame(asignaciones)

# Mostrar las primeras filas
print(df_asignaciones)

# Exportar a Excel
df_asignaciones.to_excel("asignaciones_colaboradores.xlsx", index=False)

print("Archivo exportado correctamente.")

    Colaborador        Día   Turno  Mercado
0            C1      Lunes  Mañana     Asia
1            C1     Martes   Tarde     Asia
2            C1  Miércoles  Mañana     Asia
3            C1    Viernes   Tarde     Asia
4            C1     Sábado  Mañana   Europa
..          ...        ...     ...      ...
355         C60  Miércoles   Noche     Asia
356         C60     Jueves   Noche  America
357         C60    Viernes   Noche     Asia
358         C60     Sábado   Noche   Europa
359         C60    Domingo   Noche   Europa

[360 rows x 4 columns]
Archivo exportado correctamente.


In [33]:
resumen = []

for i in model.C:

    dias_trabajados = 0

    for d in model.D:

        # ¿Trabajó ese día?
        trabajo = sum(
            value(model.x[i, d, t, m])
            for t in model.T
            for m in model.M
        )

        if trabajo > 0.5:
            dias_trabajados += 1

    resumen.append({
        "Colaborador": i,
        "Días trabajados": dias_trabajados,
        "Días descanso": len(model.D) - dias_trabajados
    })

df_resumen = pd.DataFrame(resumen)

print(df_resumen)

# Exportar a Excel
df_resumen.to_excel("resumen_dias_trabajados.xlsx", index=False)

   Colaborador  Días trabajados  Días descanso
0           C1                6              1
1           C2                6              1
2           C3                6              1
3           C4                6              1
4           C5                6              1
5           C6                6              1
6           C7                6              1
7           C8                6              1
8           C9                6              1
9          C10                6              1
10         C11                6              1
11         C12                6              1
12         C13                6              1
13         C14                6              1
14         C15                6              1
15         C16                6              1
16         C17                6              1
17         C18                6              1
18         C19                6              1
19         C20                6              1
20         C2